# Photo-z estimation with UMAP

In [1]:
import os, socket
print("hostname:    ", socket.gethostname())
print("SLURM_JOB_ID:", os.environ.get("SLURM_JOB_ID", "NOT SET — you're on a login node"))
print("SLURM_CPUS:  ", os.environ.get("SLURM_CPUS_ON_NODE", "?"))

hostname:     nid004152
SLURM_JOB_ID: 55028603
SLURM_CPUS:   256


In [2]:
### Imports
from rail.core.data import DataStore
from rail.core.stage import RailStage
from rail.core.data import PqHandle

import numpy as np
import h5py
import tables_io

from UMAPEstimator import UMAPEstimator

# Specify paths and parameters

In [21]:
### Specify path to noisy catalog
noisy_catalog_path = "/pscratch/sd/s/sajkov/data/integrated_catalog_23apr26_noised_19Jun26.pq"

### Specify path to output photometry
import time
date = time.strftime('%d%b%y', time.localtime())
photoz_path = noisy_catalog_path.split('.pq')[0] + f'_UMAPphotoz_{date}.pq'

In [22]:
### Specify model parameters
training_fraction = 0.8


metric            = "manhattan_weighted_linear"
seed              = 42

# Split data into training and validation

In [23]:
data = tables_io.read(noisy_catalog_path)

# data_cut = int(1e5)
data_cut = -1
data = data[:data_cut]

training_indices = np.zeros(len(data), dtype = bool)
training_indices[np.random.choice(len(data), size = int(training_fraction * len(data)),
                 replace = False)] = True

bands = [key for key in data.keys() if (not key.endswith('_err')) & (key != 'Roman_F146')]
error_bands = [key for key in data.keys() if key.endswith('_err')]

validation_data = data[~training_indices]
training_data   = data[training_indices]

column_list None


In [24]:
photometry_bands = [key for key in training_data.keys()\
                        if (not key.endswith('_err')) and (key != 'Roman_F146')]
phot_error_bands = [f"{key}_err" for key in photometry_bands]

In [25]:
redshift_filepath = '/pscratch/sd/s/sajkov/data/mock_catalog_Ch1_26.h5'
redshift          = h5py.File(redshift_filepath)['sfh_parameters'][:, -1]

training_redshift   = redshift[:data_cut][training_indices]
validation_redshift = redshift[:data_cut][~training_indices]

# Instantiate and run RAIL stage

In [26]:
RailStage.data_store = DataStore()

stage = UMAPEstimator.make_stage(
    name = "UMAP_informer",
    
    ambient_metric_umap = metric,
    
    seed = seed
)

stage.set_data("training_photometry", data = training_data[photometry_bands])
stage.set_data("training_phot_error", data = training_data[phot_error_bands])
stage.set_data("training_redshift",   data = training_redshift)

2026-06-25 15:14:56 [info     ] Inserting handle into data store.  training_photometry: None, UMAP_informer
2026-06-25 15:14:56 [info     ] Inserting handle into data store.  training_phot_error: None, UMAP_informer
2026-06-25 15:14:56 [info     ] Inserting handle into data store.  training_redshift: None, UMAP_informer


array([0.1092879 , 0.17016256, 0.13377383, ..., 0.10240896, 0.2242853 ,
       0.15642053], shape=(1599999,), dtype=float32)

# Inform the estimator

In [27]:
stage.UMAP_informer()

/global/homes/s/sajkov/.conda/envs/rail/lib/python3.14/site-packages/umap/umap_.py:1857: UserWarning: custom distance metric does not return gradient; inverse_transform will be unavailable. To enable using inverse_transform method, define a distance function that returns a tuple of (distance [float], gradient [np.array])
  warn(
/global/homes/s/sajkov/.conda/envs/rail/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out

2026-06-25 16:39:09 [info     ] Inserting handle into data store.  informed_reducer_UMAP_informer: inprogress_informed_reducer_UMAP_informer.pkl, UMAP_informer
2026-06-25 16:39:09 [info     ] Inserting handle into data store.  informed_embedding_UMAP_informer: inprogress_informed_embedding_UMAP_informer.pq, UMAP_informer
2026-06-25 16:39:10 [info     ] Inserting handle into data store.  informed_kNN_regressor_UMAP_informer: inprogress_informed_kNN_regressor_UMAP_informer.pkl, UMAP_informer


# Estimate redshifts

In [ ]:
stage.set_data("estimation_photometry", data = validation_data[photometry_bands])
stage.set_data("estimation_phot_error", data = validation_data[phot_error_bands])
stage.UMAP_estimator()

In [ ]:
estimated_photoz_pdfs = stage.get_handle('estimated_photoz_pdfs').data
estimated_photoz_medians = stage.get_handle('estimated_photoz_medians').data

# Evaluate

# Plot redshifts

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('/Users/leo/Projects/LBG_cosmology/code/umap_nz_cal.mplstyle')

In [ ]:
fig = plt.figure()
ax = fig.add_subplot()
ax.set_xlim(0, .5)

index = 6500

estimated_photoz_pdfs.plot(index, axes = ax)
ax.axvline(estimated_photoz_medians.values[index])

ax.set_xlabel('Redshift')
ax.set_ylabel('Density')

In [ ]:
fig = plt.figure()
ax = fig.add_subplot()


ax.hexbin(validation_redshift, estimated_photoz_medians,
          mincnt = 1)
ax.axline((0, 0), slope = 1, color = 'red')

In [ ]:
# Save otputs